In [1]:
from pathlib import Path

import pandas as pd

pasta_cnes = Path(
    "/mnt/c/Users/emanu/Downloads/BASE_DE_DADOS_CNES_202610"
)

arquivos = [
    "tbEstabelecimento202610.csv",
    "tbEquipe202610.csv",
    "rlEstabEquipeProf202610.csv",
    "tbTipoEquipe202610.csv",
    "tbTipoUnidade202610.csv",
    "tbTipoEstabelecimento202610.csv",
    "tbMunicipio202610.csv",
    "tbMotivoDesativacao202610.csv",
    "tbMotivoDesativEquipe202610.csv",
]

print("Pasta encontrada:", pasta_cnes.is_dir())

for nome in arquivos:
    caminho = pasta_cnes / nome
    print(f"{nome}: {'OK' if caminho.is_file() else 'NÃO ENCONTRADO'}")

Pasta encontrada: True
tbEstabelecimento202610.csv: OK
tbEquipe202610.csv: OK
rlEstabEquipeProf202610.csv: OK
tbTipoEquipe202610.csv: OK
tbTipoUnidade202610.csv: OK
tbTipoEstabelecimento202610.csv: OK
tbMunicipio202610.csv: OK
tbMotivoDesativacao202610.csv: OK
tbMotivoDesativEquipe202610.csv: OK


In [2]:
# Identifica uma codificação que consiga ler os dois arquivos.
nomes_dominios = [
    "tbMunicipio202610.csv",
    "tbTipoEquipe202610.csv",
]

try:
    for nome in nomes_dominios:
        (pasta_cnes / nome).read_text(encoding="utf-8-sig")
    encoding_cnes = "utf-8-sig"
except UnicodeDecodeError:
    encoding_cnes = "cp1252"

municipios = pd.read_csv(
    pasta_cnes / nomes_dominios[0],
    sep=";",
    encoding=encoding_cnes,
    dtype="string",
)

tipos_equipe = pd.read_csv(
    pasta_cnes / nomes_dominios[1],
    sep=";",
    encoding=encoding_cnes,
    dtype="string",
)

salvador = municipios.loc[
    municipios["NO_MUNICIPIO"].str.strip().str.upper().eq("SALVADOR"),
    ["CO_MUNICIPIO", "NO_MUNICIPIO", "CO_SIGLA_ESTADO"],
]

print("Codificação utilizada:", encoding_cnes)

print("\nMunicípio:")
print(salvador.to_string(index=False))

print("\nTipos de equipe:")
print(
    tipos_equipe.sort_values("TP_EQUIPE").to_string(index=False)
)

Codificação utilizada: cp1252

Município:
CO_MUNICIPIO NO_MUNICIPIO CO_SIGLA_ESTADO
      292740     SALVADOR              BA

Tipos de equipe:
TP_EQUIPE                                                    DS_EQUIPE CO_GRUPO_EQUIPE
       01                             ESF - EQUIPE DE SAUDE DA FAMILIA              01
       02                         ESFSB_M1 - ESF COM SAUDE BUCAL - M I              01
       03                        ESFSB_M2 - ESF COM SAUDE BUCAL - M II              01
       04               EACS - EQUIPE DE AGENTES COMUNITARIOS DE SAUDE              02
       05               EPEN - EQUIPE DE AT. SAUDE SIST. PENITENCIARIO              03
       06 NASF1 - NUCLEO DE APOIO A SAUDE DA FAMILIA-NASF MODALIDADE 1              04
       07 NASF2 - NUCLEO DE APOIO A SAUDE DA FAMILIA-NASF MODALIDADE 2              04
       08             EMSI - EQUIPE MULTIDISCIPLINAR DE SAUDE INDIGENA              16
       09 EMSIAL  EQ MULTIDISC AT BASICA SAUDE INDIGENA-AMAZONIA LEGAL   

In [3]:
colunas_equipes = [
    "CO_MUNICIPIO",
    "CO_AREA",
    "SEQ_EQUIPE",
    "CO_EQUIPE",
    "CO_UNIDADE",
    "TP_EQUIPE",
    "NO_REFERENCIA",
    "DT_ATIVACAO",
    "DT_DESATIVACAO",
    "CD_MOTIVO_DESATIV",
]

partes_salvador = []

with pd.read_csv(
    pasta_cnes / "tbEquipe202610.csv",
    sep=";",
    encoding=encoding_cnes,
    dtype="string",
    usecols=colunas_equipes,
    chunksize=50_000,
) as leitor:
    for bloco in leitor:
        bloco = bloco.apply(lambda coluna: coluna.str.strip())
        selecao = bloco.loc[
            bloco["CO_MUNICIPIO"].eq("292740")
        ].copy()

        if not selecao.empty:
            partes_salvador.append(selecao)

if not partes_salvador:
    raise ValueError("Nenhum registro de equipe encontrado para Salvador.")

equipes_salvador = pd.concat(partes_salvador, ignore_index=True)

contagem_tipos = (
    equipes_salvador.groupby("TP_EQUIPE", dropna=False)
    .size()
    .rename("registros")
    .reset_index()
    .merge(
        tipos_equipe[["TP_EQUIPE", "DS_EQUIPE"]],
        on="TP_EQUIPE",
        how="left",
        validate="many_to_one",
    )
    .sort_values("TP_EQUIPE")
)

print("Registros de equipes em Salvador:", len(equipes_salvador))
print("\nRegistros por tipo:")
print(contagem_tipos.to_string(index=False))

print("\nExemplos das datas e do motivo de desativação:")
print(
    equipes_salvador[
        ["DT_ATIVACAO", "DT_DESATIVACAO", "CD_MOTIVO_DESATIV"]
    ]
    .drop_duplicates()
    .head(12)
    .to_string(index=False)
)

Registros de equipes em Salvador: 1080

Registros por tipo:
TP_EQUIPE  registros                                                    DS_EQUIPE
       01          8                             ESF - EQUIPE DE SAUDE DA FAMILIA
       02         12                         ESFSB_M1 - ESF COM SAUDE BUCAL - M I
       04         36               EACS - EQUIPE DE AGENTES COMUNITARIOS DE SAUDE
       05          4               EPEN - EQUIPE DE AT. SAUDE SIST. PENITENCIARIO
       06          1 NASF1 - NUCLEO DE APOIO A SAUDE DA FAMILIA-NASF MODALIDADE 1
       22         28    EMAD I - EQUIPE MULTIPROFISSIONAL DE ATENCAO DOMICILIAR I
       23          5                      EMAP - EQUIPE MULTIDISCIPLINAR DE APOIO
       42          3     eCR MIII - EQUIPE DOS CONSULTORIOS NA RUA MODALIDADE III
       46          6  EMAD II - EQUIPE MULTIPROFISSIONAL DE ATENCAO DOMICILIAR II
       70        399                             ESF - EQUIPE DE SAUDE DA FAMILIA
       71        417                  

In [4]:
print("=== Identificação das equipes ===")

for coluna in ["CO_EQUIPE", "CO_UNIDADE"]:
    valores = equipes_salvador[coluna]
    ausentes = valores.isna() | valores.eq("").fillna(False)

    print(f"{coluna} — ausentes ou vazios: {ausentes.sum()}")

ids = equipes_salvador["CO_EQUIPE"]
ids_validos = ids.loc[ids.notna() & ids.ne("")]

print("Códigos CO_EQUIPE distintos:", ids_validos.nunique())
print(
    "Registros envolvidos em CO_EQUIPE repetido:",
    ids_validos.duplicated(keep=False).sum(),
)

chave_composta = ["CO_MUNICIPIO", "CO_AREA", "SEQ_EQUIPE"]

print(
    "Registros com chave composta incompleta:",
    (
        equipes_salvador[chave_composta].isna()
        | equipes_salvador[chave_composta].eq("")
    ).any(axis=1).sum(),
)

print(
    "Registros envolvidos em chave composta repetida:",
    equipes_salvador.duplicated(
        subset=chave_composta, keep=False
    ).sum(),
)

print("\n=== Verificação das datas ===")

for coluna in ["DT_ATIVACAO", "DT_DESATIVACAO"]:
    texto = equipes_salvador[coluna].replace("", pd.NA)

    datas = pd.to_datetime(
        texto,
        format="%d/%m/%Y",
        errors="coerce",
    )

    invalidas = texto.notna() & datas.isna()

    print(f"\n{coluna}:")
    print("Sem preenchimento:", texto.isna().sum())
    print("Preenchidas, mas não interpretadas:", invalidas.sum())
    print("Menor data:", datas.min())
    print("Maior data:", datas.max())

    if invalidas.any():
        print("Valores a investigar:")
        print(texto.loc[invalidas].value_counts().to_string())

=== Identificação das equipes ===
CO_EQUIPE — ausentes ou vazios: 0
CO_UNIDADE — ausentes ou vazios: 0
Códigos CO_EQUIPE distintos: 1080
Registros envolvidos em CO_EQUIPE repetido: 0
Registros com chave composta incompleta: 0
Registros envolvidos em chave composta repetida: 0

=== Verificação das datas ===

DT_ATIVACAO:
Sem preenchimento: 0
Preenchidas, mas não interpretadas: 0
Menor data: 0199-11-15 00:00:00
Maior data: 2026-05-21 00:00:00

DT_DESATIVACAO:
Sem preenchimento: 980
Preenchidas, mas não interpretadas: 0
Menor data: 2011-08-17 00:00:00
Maior data: 2026-10-21 00:00:00


In [ ]:
datas_ativacao = pd.to_datetime(
    equipes_salvador["DT_ATIVACAO"],
    format="%d/%m/%Y",
    errors="coerce",
)

# Limite usado apenas para investigar datas suspeitas.
suspeitas = datas_ativacao.dt.year.lt(1900)

print("Equipes com ativação anterior a 1900:", suspeitas.sum())

print(
    equipes_salvador.loc[
        suspeitas,
        [
            "CO_EQUIPE",
            "CO_UNIDADE",
            "TP_EQUIPE",
            "NO_REFERENCIA",
            "DT_ATIVACAO",
            "DT_DESATIVACAO",
        ],
    ].to_string(index=False)
)

print("\nMotivos de desativação registrados:")
print(
    equipes_salvador["CD_MOTIVO_DESATIV"]
    .fillna("(ausente)")
    .value_counts()
    .to_string()
)

Equipes com ativação anterior a 1900: 2
 CO_EQUIPE    CO_UNIDADE TP_EQUIPE               NO_REFERENCIA DT_ATIVACAO DT_DESATIVACAO
0000210854 2927400006785        04             MARECHAL RONDON  15/11/0199     30/09/2015
0000211621 2927402653427        04 PLATAFORMA / ALTO DO BARIRI  01/01/0998     31/03/2016

Motivos de desativação registrados:
CD_MOTIVO_DESATIV
(ausente)    980
02            47
01            26
04            16
09             8
08             1
07             1
03             1


In [6]:
referencia = pd.Timestamp("2026-10-31")

# Trabalha em uma cópia, preservando a tabela original.
diagnostico_esf = equipes_salvador.loc[
    equipes_salvador["TP_EQUIPE"].isin(["01", "02", "70"])
].copy()

ativacao = pd.to_datetime(
    diagnostico_esf["DT_ATIVACAO"],
    format="%d/%m/%Y",
    errors="coerce",
)

texto_desativacao = diagnostico_esf["DT_DESATIVACAO"].replace("", pd.NA)

desativacao = pd.to_datetime(
    texto_desativacao,
    format="%d/%m/%Y",
    errors="coerce",
)

motivo = diagnostico_esf["CD_MOTIVO_DESATIV"].replace("", pd.NA)

diagnostico_esf["situacao_referencia"] = "A conferir"

datas_consistentes = (
    ativacao.notna()
    & ativacao.dt.year.ge(1900)
    & (texto_desativacao.isna() | desativacao.notna())
    & (desativacao.isna() | desativacao.ge(ativacao))
)

diagnostico_esf.loc[
    datas_consistentes & ativacao.gt(referencia),
    "situacao_referencia",
] = "Ativação posterior à referência"

diagnostico_esf.loc[
    datas_consistentes
    & ativacao.le(referencia)
    & desativacao.le(referencia),
    "situacao_referencia",
] = "Desativada até a referência"

diagnostico_esf.loc[
    datas_consistentes
    & ativacao.le(referencia)
    & (
        (texto_desativacao.isna() & motivo.isna())
        | desativacao.gt(referencia)
    ),
    "situacao_referencia",
] = "Sem desativação até a referência"

print(
    pd.crosstab(
        diagnostico_esf["TP_EQUIPE"],
        diagnostico_esf["situacao_referencia"],
        margins=True,
    ).to_string()
)

pendencias = diagnostico_esf.loc[
    diagnostico_esf["situacao_referencia"].eq("A conferir")
]

print("\nRegistros a conferir:", len(pendencias))

if not pendencias.empty:
    print(
        pendencias[
            [
                "CO_EQUIPE", "TP_EQUIPE", "DT_ATIVACAO",
                "DT_DESATIVACAO", "CD_MOTIVO_DESATIV",
            ]
        ].to_string(index=False)
    )

situacao_referencia  Desativada até a referência  Sem desativação até a referência  All
TP_EQUIPE                                                                              
01                                             8                                 0    8
02                                            12                                 0   12
70                                             7                               392  399
All                                           27                               392  419

Registros a conferir: 0


In [7]:
equipes_esf_candidatas = diagnostico_esf.loc[
    diagnostico_esf["situacao_referencia"].eq(
        "Sem desativação até a referência"
    )
].copy()

equipes_por_unidade = (
    equipes_esf_candidatas.groupby("CO_UNIDADE")
    .agg(total_esf=("CO_EQUIPE", "nunique"))
    .reset_index()
    .sort_values(
        ["total_esf", "CO_UNIDADE"],
        ascending=[False, True],
    )
)

print("ESF candidatas:", len(equipes_esf_candidatas))
print("Estabelecimentos vinculados:", len(equipes_por_unidade))
print("Soma das equipes por estabelecimento:",
      equipes_por_unidade["total_esf"].sum())

print("\nDez estabelecimentos com mais equipes:")
print(equipes_por_unidade.head(10).to_string(index=False))

ESF candidatas: 392
Estabelecimentos vinculados: 119
Soma das equipes por estabelecimento: 392

Dez estabelecimentos com mais equipes:
   CO_UNIDADE  total_esf
2927407594003          6
2927409006710          6
2927400004014          5
2927400006807          5
2927407509553          5
2927407573960          5
2927400004022          4
2927400004618          4
2927400004898          4
2927400005231          4


In [8]:
colunas_estabelecimentos = [
    "CO_UNIDADE",
    "CO_CNES",
    "NO_FANTASIA",
    "NO_LOGRADOURO",
    "NU_ENDERECO",
    "NO_BAIRRO",
    "TP_UNIDADE",
    "CO_TIPO_UNIDADE",
    "CO_TIPO_ESTABELECIMENTO",
    "CO_MUNICIPIO_GESTOR",
    "CO_MOTIVO_DESAB",
    "NU_LATITUDE",
    "NU_LONGITUDE",
]

ids_unidades_esf = set(equipes_por_unidade["CO_UNIDADE"])
partes_estabelecimentos = []

with pd.read_csv(
    pasta_cnes / "tbEstabelecimento202610.csv",
    sep=";",
    encoding=encoding_cnes,
    dtype="string",
    usecols=colunas_estabelecimentos,
    chunksize=50_000,
) as leitor:
    for bloco in leitor:
        bloco = bloco.apply(lambda coluna: coluna.str.strip())

        selecao = bloco.loc[
            bloco["CO_UNIDADE"].isin(ids_unidades_esf)
        ].copy()

        if not selecao.empty:
            partes_estabelecimentos.append(selecao)

if not partes_estabelecimentos:
    raise ValueError("Nenhum estabelecimento vinculado foi encontrado.")

estabelecimentos_esf = pd.concat(
    partes_estabelecimentos,
    ignore_index=True,
)

encontrados = set(estabelecimentos_esf["CO_UNIDADE"])
faltantes = ids_unidades_esf - encontrados

print("Registros encontrados:", len(estabelecimentos_esf))
print("Estabelecimentos distintos:", len(encontrados))
print("Estabelecimentos não encontrados:", len(faltantes))
print(
    "Registros com CO_UNIDADE repetido:",
    estabelecimentos_esf["CO_UNIDADE"].duplicated(keep=False).sum(),
)

if faltantes:
    print("Códigos não encontrados:", sorted(faltantes))

print("\nClassificações encontradas:")
print(
    estabelecimentos_esf.groupby(
        ["TP_UNIDADE", "CO_TIPO_UNIDADE", "CO_TIPO_ESTABELECIMENTO"],
        dropna=False,
    )
    .size()
    .rename("registros")
    .to_string()
)

print("\nMotivos de desativação/desabilitação registrados:")
print(
    estabelecimentos_esf["CO_MOTIVO_DESAB"]
    .replace("", pd.NA)
    .fillna("(ausente)")
    .value_counts()
    .to_string()
)

Registros encontrados: 119
Estabelecimentos distintos: 119
Estabelecimentos não encontrados: 0
Registros com CO_UNIDADE repetido: 0

Classificações encontradas:
TP_UNIDADE  CO_TIPO_UNIDADE  CO_TIPO_ESTABELECIMENTO
02          <NA>             001                        119

Motivos de desativação/desabilitação registrados:
CO_MOTIVO_DESAB
(ausente)    119


In [9]:
tipos_unidade = pd.read_csv(
    pasta_cnes / "tbTipoUnidade202610.csv",
    sep=";",
    encoding=encoding_cnes,
    dtype="string",
)

tipos_estabelecimento = pd.read_csv(
    pasta_cnes / "tbTipoEstabelecimento202610.csv",
    sep=";",
    encoding=encoding_cnes,
    dtype="string",
)

print("Descrição candidata para TP_UNIDADE = 02:")
print(
    tipos_unidade.loc[
        tipos_unidade["CO_TIPO_UNIDADE"].str.strip().eq("02")
    ].to_string(index=False)
)

print("\nDescrição de CO_TIPO_ESTABELECIMENTO = 001:")
print(
    tipos_estabelecimento.loc[
        tipos_estabelecimento["CO_TIPO_ESTABELECIMENTO"]
        .str.strip().eq("001")
    ].to_string(index=False)
)

Descrição candidata para TP_UNIDADE = 02:
CO_TIPO_UNIDADE                DS_TIPO_UNIDADE
             02 CENTRO DE SAUDE/UNIDADE BASICA

Descrição de CO_TIPO_ESTABELECIMENTO = 001:
CO_TIPO_ESTABELECIMENTO DS_TIPO_ESTABELECIMENTO DS_CONCEITO_TIPO
                    001 UNIDADE BASICA DE SAUDE             <NA>


In [10]:
coordenadas_esf = estabelecimentos_esf.copy()

for origem, destino in [
    ("NU_LATITUDE", "latitude"),
    ("NU_LONGITUDE", "longitude"),
]:
    texto = (
        coordenadas_esf[origem]
        .str.strip()
        .replace("", pd.NA)
        .str.replace(",", ".", regex=False)
    )

    coordenadas_esf[destino] = pd.to_numeric(
        texto,
        errors="coerce",
    )

    print(f"\n{origem}:")
    print("Ausentes:", texto.isna().sum())
    print(
        "Preenchidas, mas não numéricas:",
        (texto.notna() & coordenadas_esf[destino].isna()).sum(),
    )

lat = coordenadas_esf["latitude"]
lon = coordenadas_esf["longitude"]

coordenadas_esf["coordenada_global_valida"] = (
    lat.between(-90, 90)
    & lon.between(-180, 180)
    & lat.ne(0)
    & lon.ne(0)
).fillna(False)

print("\nResumo das coordenadas:")
print(coordenadas_esf[["latitude", "longitude"]].describe().to_string())

print(
    "\nEstabelecimentos com coordenadas ausentes ou inválidas:",
    (~coordenadas_esf["coordenada_global_valida"]).sum(),
)


NU_LATITUDE:
Ausentes: 0
Preenchidas, mas não numéricas: 0

NU_LONGITUDE:
Ausentes: 0
Preenchidas, mas não numéricas: 0

Resumo das coordenadas:
        latitude  longitude
count      119.0      119.0
mean  -12.928703 -38.466563
std     0.063524   0.065938
min   -13.107777 -38.889134
25%    -12.97196 -38.488036
50%   -12.933149 -38.473842
75%   -12.897335 -38.441473
max   -12.699638 -38.333094

Estabelecimentos com coordenadas ausentes ou inválidas: 0


In [ ]:
import geopandas as gpd


raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

pref_bairro_cnes = gpd.read_file(
    raiz / "dados/brutos/bairros/prefeitura_bairro.gpkg",
    layer="prefeituras_bairro_a",
)[["COD_PBR", "PREF_BAIRR", "geometry"]].to_crs(epsg=31984)

pontos_estabelecimentos_esf = gpd.GeoDataFrame(
    coordenadas_esf.copy(),
    geometry=gpd.points_from_xy(
        coordenadas_esf["longitude"].astype(float),
        coordenadas_esf["latitude"].astype(float),
    ),
    crs="EPSG:4326",
).to_crs(pref_bairro_cnes.crs)

associacao_esf = gpd.sjoin(
    pontos_estabelecimentos_esf,
    pref_bairro_cnes,
    how="left",
    predicate="intersects",
)

quantidade_associacoes = (
    associacao_esf.groupby("CO_UNIDADE")["COD_PBR"]
    .count()
)

print("Estabelecimentos por quantidade de regiões encontradas:")
print(
    quantidade_associacoes.value_counts()
    .sort_index()
    .rename_axis("regioes_encontradas")
    .rename("estabelecimentos")
    .to_string()
)

sem_regiao = associacao_esf.loc[
    associacao_esf["COD_PBR"].isna()
]

print("\nEstabelecimentos sem região:", len(sem_regiao))

if not sem_regiao.empty:
    print(
        sem_regiao[
            [
                "CO_CNES", "NO_FANTASIA", "NO_BAIRRO",
                "latitude", "longitude",
            ]
        ].to_string(index=False)
    )

Estabelecimentos por quantidade de regiões encontradas:
regioes_encontradas
0      3
1    116

Estabelecimentos sem região: 3
CO_CNES                NO_FANTASIA      NO_BAIRRO   latitude  longitude
9749896      USF VILA FRATERNIDADE FAZENDA COUTOS -12.897621 -38.677044
0053465 USF CAJAZEIRAS JAGUARIPE I      JAGUARIPE -13.107777 -38.889134
3270505           USF VISTA ALEGRE   VISTA ALEGRE -12.699638 -38.333094


In [12]:
pendencias_localizacao = (
    sem_regiao.drop(columns=["index_right"], errors="ignore")
    .merge(
        equipes_por_unidade,
        on="CO_UNIDADE",
        how="left",
        validate="one_to_one",
    )
)

print("Unidades com localização pendente:")
print(
    pendencias_localizacao[
        ["CO_CNES", "NO_FANTASIA", "NO_BAIRRO", "total_esf"]
    ].to_string(index=False)
)

print(
    "\nESF vinculadas às unidades pendentes:",
    pendencias_localizacao["total_esf"].sum(),
)

unidades_anteriores = gpd.read_file(
    raiz / "dados/processados/ubs_usf_salvador.gpkg",
    layer="ubs_usf",
)

# Busca apenas candidatos por nome; não estabelece correspondência.
candidatas = unidades_anteriores.loc[
    unidades_anteriores["unidade"].str.contains(
        r"FRATERNIDADE|JAGUARIPE|VISTA\s+ALEGRE",
        case=False,
        na=False,
        regex=True,
    )
].copy()

print("\nCandidatas na base anterior:")
print(
    candidatas[
        ["unidade", "bairro", "logradouro", "numero", "tipo_tratado"]
    ].to_string(index=False)
)

print("\nEndereços registrados no CNES:")
print(
    pendencias_localizacao[
        ["CO_CNES", "NO_FANTASIA", "NO_LOGRADOURO", "NU_ENDERECO"]
    ].to_string(index=False)
)

Unidades com localização pendente:
CO_CNES                NO_FANTASIA      NO_BAIRRO  total_esf
9749896      USF VILA FRATERNIDADE FAZENDA COUTOS          4
0053465 USF CAJAZEIRAS JAGUARIPE I      JAGUARIPE          3
3270505           USF VISTA ALEGRE   VISTA ALEGRE          2

ESF vinculadas às unidades pendentes: 9

Candidatas na base anterior:
                 unidade       bairro                logradouro numero tipo_tratado
        USF VISTA ALEGRE VISTA ALEGRE                 RUA SABIA    S/N          USF
USF VILA DA FRATERNIDADE       COUTOS RUA JOSE DO PATROCINIO \n  S/N\n          USF

Endereços registrados no CNES:
CO_CNES                NO_FANTASIA          NO_LOGRADOURO NU_ENDERECO
9749896      USF VILA FRATERNIDADE RUA JOSE DO PATROCINIO         S/N
0053465 USF CAJAZEIRAS JAGUARIPE I                  RUA F         S/N
3270505           USF VISTA ALEGRE              RUA SABIA          57


In [13]:
candidatas_pb = gpd.sjoin(
    candidatas.to_crs(pref_bairro_cnes.crs),
    pref_bairro_cnes,
    how="left",
    predicate="intersects",
)

print("Localização das candidatas da base anterior:")
print(
    candidatas_pb[
        ["unidade", "logradouro", "numero", "COD_PBR", "PREF_BAIRR"]
    ].to_string(index=False)
)

# Busca ampliada: possíveis nomes ou endereços relacionados a Jaguaripe.
nomes = (
    unidades_anteriores["unidade"]
    .fillna("")
    .str.upper()
)

bairros = (
    unidades_anteriores["bairro"]
    .fillna("")
    .str.upper()
)

logradouros = (
    unidades_anteriores["logradouro"]
    .fillna("")
    .str.upper()
)

busca_jaguaripe = (
    nomes.str.contains("JAGUARIPE", regex=False)
    | bairros.str.contains("JAGUARIPE", regex=False)
    | logradouros.str.contains(
        r"\bRUA\s+F\b",
        regex=True,
    )
)

print("\nPossíveis candidatas para Jaguaripe:")
print(
    unidades_anteriores.loc[
        busca_jaguaripe,
        ["unidade", "bairro", "logradouro", "numero", "tipo_tratado"],
    ].to_string(index=False)
)

Localização das candidatas da base anterior:
                 unidade                logradouro numero COD_PBR                              PREF_BAIRR
        USF VISTA ALEGRE                 RUA SABIA    S/N      II Prefeitura-bairro II - Subúrbio / Ilhas
USF VILA DA FRATERNIDADE RUA JOSE DO PATROCINIO \n  S/N\n      II Prefeitura-bairro II - Subúrbio / Ilhas

Possíveis candidatas para Jaguaripe:
      unidade      bairro                  logradouro numero tipo_tratado
USF JAGUARIBE JAGUARIPE I CAMINIO 09G JAGUARIPE L2-44    S/N          USF


In [14]:
from IPython.display import display

pontos_cnes = pendencias_localizacao.to_crs(epsg=4326)

candidatas_anteriores = unidades_anteriores.loc[
    unidades_anteriores["unidade"].str.contains(
        r"FRATERNIDADE|VISTA\s+ALEGRE|JAGUARIBE",
        case=False,
        na=False,
        regex=True,
    )
].copy()

candidatas_anteriores = gpd.sjoin(
    candidatas_anteriores.to_crs(pref_bairro_cnes.crs),
    pref_bairro_cnes[["COD_PBR", "geometry"]],
    how="left",
    predicate="intersects",
).to_crs(epsg=4326)

def link_osm(ponto):
    return (
        "https://www.openstreetmap.org/"
        f"?mlat={ponto.y:.7f}&mlon={ponto.x:.7f}"
        f"#map=18/{ponto.y:.7f}/{ponto.x:.7f}"
    )

linhas = []

for _, registro in pontos_cnes.iterrows():
    linhas.append({
        "base": "CNES 202610",
        "nome": registro["NO_FANTASIA"],
        "regiao": "Fora do recorte",
        "mapa": link_osm(registro.geometry),
    })

for _, registro in candidatas_anteriores.iterrows():
    linhas.append({
        "base": "Base anterior",
        "nome": registro["unidade"],
        "regiao": registro["COD_PBR"],
        "mapa": link_osm(registro.geometry),
    })

display(pd.DataFrame(linhas))

,base,nome,regiao,mapa
0,CNES 202610,USF VILA FRATERNIDADE,Fora do recorte,https://www.openstreetmap.org/?mlat=-12.897621...
1,CNES 202610,USF CAJAZEIRAS JAGUARIPE I,Fora do recorte,https://www.openstreetmap.org/?mlat=-13.107776...
2,CNES 202610,USF VISTA ALEGRE,Fora do recorte,https://www.openstreetmap.org/?mlat=-12.699638...
3,Base anterior,USF JAGUARIBE,III,https://www.openstreetmap.org/?mlat=-12.903648...
4,Base anterior,USF VISTA ALEGRE,II,https://www.openstreetmap.org/?mlat=-12.851819...
5,Base anterior,USF VILA DA FRATERNIDADE,II,https://www.openstreetmap.org/?mlat=-12.847876...


In [15]:
from IPython.display import Markdown, display

for linha in linhas:
    display(Markdown(
        f"**{linha['base']} — {linha['nome']}** "
        f"(região: {linha['regiao']}) — "
        f"[Abrir no mapa]({linha['mapa']})"
    ))

**CNES 202610 — USF VILA FRATERNIDADE** (região: Fora do recorte) — [Abrir no mapa](https://www.openstreetmap.org/?mlat=-12.8976210&mlon=-38.6770440#map=18/-12.8976210/-38.6770440)

**CNES 202610 — USF CAJAZEIRAS JAGUARIPE I** (região: Fora do recorte) — [Abrir no mapa](https://www.openstreetmap.org/?mlat=-13.1077766&mlon=-38.8891339#map=18/-13.1077766/-38.8891339)

**CNES 202610 — USF VISTA ALEGRE** (região: Fora do recorte) — [Abrir no mapa](https://www.openstreetmap.org/?mlat=-12.6996382&mlon=-38.3330944#map=18/-12.6996382/-38.3330944)

**Base anterior — USF JAGUARIBE** (região: III) — [Abrir no mapa](https://www.openstreetmap.org/?mlat=-12.9036488&mlon=-38.4051160#map=18/-12.9036488/-38.4051160)

**Base anterior — USF VISTA ALEGRE** (região: II) — [Abrir no mapa](https://www.openstreetmap.org/?mlat=-12.8518196&mlon=-38.4615607#map=18/-12.8518196/-38.4615607)

**Base anterior — USF VILA DA FRATERNIDADE** (região: II) — [Abrir no mapa](https://www.openstreetmap.org/?mlat=-12.8478765&mlon=-38.4703306#map=18/-12.8478765/-38.4703306)

In [16]:
contagem_regional = (
    associacao_esf.loc[
        associacao_esf["COD_PBR"].notna(),
        ["CO_UNIDADE", "COD_PBR"],
    ]
    .merge(
        equipes_por_unidade,
        on="CO_UNIDADE",
        validate="one_to_one",
    )
    .groupby("COD_PBR", as_index=False)
    .agg(
        total_esf=("total_esf", "sum"),
        estabelecimentos_com_esf=("CO_UNIDADE", "nunique"),
    )
)

indicadores_anteriores = pd.read_csv(
    raiz / "resultados/tabelas/indicadores_por_prefeitura_bairro_revisado.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    dtype={"COD_PBR": "string"},
)

indicadores_esf = indicadores_anteriores[
    ["COD_PBR", "PREF_BAIRR", "pop_estimada"]
].merge(
    contagem_regional,
    on="COD_PBR",
    how="left",
    validate="one_to_one",
)

for coluna in ["total_esf", "estabelecimentos_com_esf"]:
    indicadores_esf[coluna] = (
        indicadores_esf[coluna].fillna(0).astype(int)
    )

indicadores_esf["esf_por_10_mil"] = (
    indicadores_esf["total_esf"]
    / indicadores_esf["pop_estimada"] * 10_000
)

indicadores_esf["pop_estimada_por_esf"] = (
    indicadores_esf["pop_estimada"]
    / indicadores_esf["total_esf"].replace(0, float("nan"))
)

assert indicadores_esf["total_esf"].sum() == 383

print(indicadores_esf.round(2).to_string(index=False))
print("\nESF regionalizadas: 383")
print("ESF com localização pendente: 9")
print("Total municipal candidato: 392")

COD_PBR                                       PREF_BAIRR  pop_estimada  total_esf  estabelecimentos_com_esf  esf_por_10_mil  pop_estimada_por_esf
     II          Prefeitura-bairro II - Subúrbio / Ilhas     258468.87         50                        18            1.93               5169.38
    VII  Prefeitura-bairro VII - Liberdade / São Caetano     297579.80         47                        14            1.58               6331.49
   VIII Prefeitura-bairro VIII - Cabula / Tancredo Neves     317896.44         50                        13            1.57               6357.93
      I            Prefeitura-bairro I - Centro / Brotas     238643.08         48                        14            2.01               4971.73
      V               Prefeitura-bairro V - Cidade Baixa     146910.15         16                         6            1.09               9181.88
     IX              Prefeitura-bairro IX -  Pau da Lima     173315.89         14                         4            0.81 

In [17]:
pasta_saida = raiz / "resultados/tabelas"
pasta_saida.mkdir(parents=True, exist_ok=True)

indicadores_esf.assign(
    competencia_cnes="202610",
    situacao="preliminar_com_9_equipes_sem_regiao",
).to_csv(
    pasta_saida / "esf_por_prefeitura_bairro_cnes_202610.csv",
    sep=";",
    decimal=",",
    index=False,
    encoding="utf-8-sig",
)

pendencias_localizacao[
    [
        "CO_CNES", "NO_FANTASIA", "NO_BAIRRO",
        "latitude", "longitude", "total_esf",
    ]
].to_csv(
    pasta_saida / "pendencias_localizacao_esf_cnes_202610.csv",
    sep=";",
    decimal=",",
    index=False,
    encoding="utf-8-sig",
)

nota = """Equipes de Saúde da Família — CNES, competência 202610

Foram identificadas 392 equipes distintas do tipo 70, ativadas até
31/10/2026 e sem desativação até essa data, conforme o cadastro consultado.
A situação cadastral não comprova funcionamento efetivo, composição
profissional completa ou atendimento realizado.

As equipes estão vinculadas a 119 estabelecimentos. Pelas coordenadas
do CNES, 116 estabelecimentos e 383 equipes foram associados a uma
Prefeitura-Bairro. Três estabelecimentos, reunindo nove equipes,
permanecem com localização pendente. Não foram corrigidas coordenadas.

Os indicadores regionais são preliminares e excluem essas nove equipes.
Essa exclusão reduz a razão de equipes por habitante e aumenta a razão
de população por equipe nas regiões às quais elas eventualmente pertençam.

O denominador é a população estimada no recorte das Prefeituras-Bairro,
derivada do Censo 2022. Portanto, há diferença temporal entre população
e cadastro de equipes.

População estimada por equipe é uma razão regional, não população
cadastrada, atendida ou cobertura assistencial.

Esta etapa não substituiu nem atualizou integralmente a base anterior
de 142 UBS/USF e não recalculou as distâncias ou o mapa publicado.
"""

(pasta_saida / "notas_esf_cnes_202610.txt").write_text(
    nota,
    encoding="utf-8",
)

print("Tabela regional, pendências e nota metodológica salvas.")

Tabela regional, pendências e nota metodológica salvas.
